# 대화 이력을 기억하는 RAG — 문맥을 유지하는 대화형 QA

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- `RunnableWithMessageHistory`로 대화 이력을 RAG 체인에 통합하기
- `ChatMessageHistory`와 세션 ID로 여러 사용자의 대화를 독립 관리하기
- `MessagesPlaceholder`를 프롬프트에 삽입해 이전 대화를 문맥으로 활용하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- 세션(Session): 사용자별 독립적인 대화 공간

---

```mermaid
flowchart LR
    U[사용자 질문]:::input --> H[ChatMessageHistory<br/>이전 대화 로드]:::storage
    H --> P[ChatPromptTemplate<br/>system + history + question]:::process
    R[(VectorStore)]:::storage --> P
    P --> L[LLM<br/>문맥 있는 답변]:::process
    L --> S[ChatMessageHistory<br/>답변 저장]:::storage
    L --> A[최종 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 일반 RAG vs 대화형 RAG

| | 일반 RAG | 대화형 RAG |
|--|---------|-----------|
| 질문 1 | "훈련수당이 어떻게 바뀌나요?" → 잘 답변 | "훈련수당이 어떻게 바뀌나요?" → 잘 답변 |
| 질문 2 | "변경 전 금액은?" → "변경 전"이 뭔지 모름 | "변경 전 금액은?" → 훈련수당의 변경 전 금액으로 이해 |

> **실무 팁**: 대화가 길어지면 컨텍스트 토큰이 증가해요. 최근 5~10개 메시지만 유지하거나, 오래된 대화를 요약해서 저장하는 전략을 고려해보세요.

> 🎯 **강의 포인트**: 대화형 RAG의 핵심은 **세션 관리**입니다. `session_id`로 사용자별로 대화 이력을 독립적으로 유지하는 구조를 이해하는 것이 중요합니다. 실제 서비스에서는 인메모리 `store` 딕셔너리 대신 Redis나 DB로 교체해야 합니다.

> ⚠️ **자주 하는 실수**: 대화 이력이 길어질수록 매 요청마다 전송되는 토큰 수가 선형으로 증가합니다. 100회 대화 후에는 컨텍스트 윈도우가 가득 찰 수 있습니다. **최근 N턴만 유지하거나 요약하는 전략**이 필수입니다.

## 일반 RAG vs 대화형 RAG

### 일반 RAG의 한계

```
User: 중증장애인 훈련수당이 어떻게 바뀌나요?
AI: 2026년부터 훈련비가 1일당 35,000원으로 인상됩니다.

User: 변경 전 금액은 얼마였나요?
AI: ❌ "변경 전"이 무엇을 가리키는지 모름 (새로운 독립 질문으로 처리)
```

### 대화형 RAG

```
User: 중증장애인 훈련수당이 어떻게 바뀌나요?
AI: 2026년부터 훈련비가 1일당 35,000원으로 인상됩니다.

User: 변경 전 금액은 얼마였나요?
AI: ✅ 변경 전에는 훈련준비금 40,000원 + 훈련비 18,000원/일이었습니다.
     (이전 대화에서 "변경 전" = "훈련수당 변경 전" 파악)
```

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from operator import itemgetter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

## 1. 문서 준비 및 RAG 기본 설정

In [3]:
# ---------------------------------------------------
# 문서 로드, 분할, 임베딩, 벡터스토어 초기화
# ---------------------------------------------------
# ============================================================
# TODO: PDF를 로드하고 분할한 뒤 FAISS 벡터스토어와 검색기를 만드세요
# 힌트: PyMuPDFLoader → split_documents → FAISS.from_documents → as_retriever(k=4)
# 예상 결과: "문서 및 검색기 준비 완료" 출력
# ============================================================

# 문서 로드
loader = PyMuPDFLoader("./data/2026_gov.pdf")
docs = loader.load()

# 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_documents = text_splitter.split_documents(docs[41:95])

# 벡터스토어
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", chunk_size=300)
vectorstore = FAISS.from_documents(split_documents, embeddings)

# 검색기
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ 문서 및 검색기 준비 완료")

✅ 문서 및 검색기 준비 완료


In [4]:
# ---------------------------------------------------
# 세션별 대화 이력 저장소 구성
# ---------------------------------------------------
# ============================================================
# TODO: get_session_history() 함수를 완성하여 session_id별로 이력을 반환하세요
# 힌트: store 딕셔너리에 ChatMessageHistory()를 세션별로 저장
# 예상 결과: "대화 이력 저장소 설정 완료" 출력
# ============================================================

# 세션별 대화 이력 저장소 (인메모리)
# ⚠️ 주의: 서버 재시작 시 모든 대화 이력이 사라짐 — 프로덕션에선 Redis/DB 사용
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """세션 ID에 해당하는 대화 이력을 반환합니다."""
    # 🎯 강의 포인트: 세션 ID만 다르면 완전히 독립된 대화 공간이 생성됨
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("✅ 대화 이력 저장소 설정 완료")
print("- 세션별로 독립적인 대화 이력 관리")
print("- 메모리 기반 저장 (재시작 시 초기화)")

✅ 대화 이력 저장소 설정 완료
- 세션별로 독립적인 대화 이력 관리
- 메모리 기반 저장 (재시작 시 초기화)


In [5]:
# ---------------------------------------------------
# MessagesPlaceholder로 대화 이력이 들어가는 프롬프트 구성
# ---------------------------------------------------
# ============================================================
# TODO: ChatPromptTemplate에 MessagesPlaceholder("chat_history")를 삽입하세요
# 힌트: from_messages([("system", ...), MessagesPlaceholder("chat_history"), ("human", ...)])
# 예상 결과: "대화형 프롬프트 생성 완료" 출력
# ============================================================

# 🔑 핵심 개념: MessagesPlaceholder는 런타임에 실제 메시지 리스트로 교체되는 동적 슬롯
# system → [이전 대화들] → human 순서가 LLM이 문맥을 올바르게 파악하는 핵심 구조
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥(Context)과 대화 이력(Chat History)을 참고하여 질문에 답변하세요.

대화 이력을 활용하여:
- 대명사("그것", "이것" 등)의 지시 대상 파악
- 이전 질문과 관련된 추가 정보 제공
- 문맥을 이어가는 자연스러운 답변

#Context:
{context}
"""),
    MessagesPlaceholder(variable_name="chat_history"),  # 💡 팁: variable_name은 RunnableWithMessageHistory의 history_messages_key와 일치해야 함
    ("human", "{question}")
])

print("✅ 대화형 프롬프트 생성 완료")

✅ 대화형 프롬프트 생성 완료


In [6]:
# ---------------------------------------------------
# RunnableWithMessageHistory로 대화 이력 관리를 체인에 통합
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain을 RunnableWithMessageHistory로 감싸세요
# 힌트: RunnableWithMessageHistory(rag_chain, get_session_history, input_messages_key="question", history_messages_key="chat_history")
# 예상 결과: "대화형 RAG 체인 생성 완료" 출력
# ============================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🔑 핵심: RunnableWithMessageHistory가 입력에 chat_history를 추가하므로
# RunnableParallel에서 chat_history도 함께 전달해야 프롬프트가 올바르게 동작합니다
rag_chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | conversational_prompt
    | llm
    | StrOutputParser()
)

# 🎯 강의 포인트: RunnableWithMessageHistory가 체인 호출 전후에 자동으로
# 이력을 로드하고 저장하는 래퍼(Wrapper) 역할을 합니다
# input_messages_key: 사용자 입력 키 / history_messages_key: 프롬프트 변수와 일치해야 함
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("✅ 대화형 RAG 체인 생성 완료")


✅ 대화형 RAG 체인 생성 완료


## 5. 대화형 RAG 실행

세션 ID를 지정하여 대화를 시작합니다.

In [7]:
# 세션 설정
session_id = "user_001"
config = {"configurable": {"session_id": session_id}}

# 첫 번째 질문 — 문서에 있는 구체적 정책 정보를 검색
question1 = "2026년에 바뀌는 세액공제 제도를 종류별로 정리해줘"
print(f"👤 User: {question1}")
answer1 = conversational_rag_chain.invoke({"question": question1}, config=config)
print(f"🤖 AI: {answer1}\n")

👤 User: 2026년에 바뀌는 세액공제 제도를 종류별로 정리해줘
🤖 AI: 2026년에 바뀌는 세액공제 제도는 다음과 같이 정리할 수 있습니다:

1. **중고자동차 매입세액공제 특례**
   - **주요 내용**: 중고자동차에 대한 공제한도를 신설합니다.
   - **공제한도**: 과세표준에서 세금계산서를 발급받고 매입한 매입가액을 차감.
   - **이월공제**: 2개 과세기간(1년)간 이월공제 허용.
   - **시행일**: 2026년 7월 1일 이후 개시하는 과세기간부터 적용.

2. **통합고용세액공제**
   - **주요 내용**: 공제액 구조를 개편하고 사후관리를 합리화합니다.
   - **공제액 구조**: 2단계 구조로 재설계.
   - **사후관리 방식**: 고용 유지 시 더 높은 공제 혜택을 부여하는 방식으로 전환.
   - **시행일**: 2026년 1월 1일 이후 개시하는 과세연도를 최초 공제연도로 하여 적용.

이 외에도 외국법인 연락사무소에 대한 과태료 부과 규정이 신설되며, 이는 세액공제와는 별개로 자료 제출 실효성을 확보하기 위한 조치입니다.



In [8]:
question2 = "그 중 시행일이 7월 1일 이후인건 뭐야?"
print(f"👤 User: {question2}")
answer2 = conversational_rag_chain.invoke({"question": question2}, config=config)
print(f"🤖 AI: {answer2}\n")

👤 User: 그 중 시행일이 7월 1일 이후인건 뭐야?
🤖 AI: 2026년 7월 1일 이후에 시행되는 세액공제 제도는 **중고자동차 매입세액공제 특례**입니다. 이 제도는 중고자동차에 대한 공제한도를 신설하고, 과세표준에서 세금계산서를 발급받고 매입한 매입가액을 차감하는 방식으로 운영됩니다. 또한, 이월공제를 허용하여 2개 과세기간(1년) 동안 이월할 수 있습니다.



In [9]:
question3 = "이월 공제가 허용되는 기간은?"
print(f"👤 User: {question3}")
answer3 = conversational_rag_chain.invoke({"question": question3}, config=config)
print(f"🤖 AI: {answer3}\n")

👤 User: 이월 공제가 허용되는 기간은?
🤖 AI: 이월 공제가 허용되는 기간은 **2개 과세기간(1년)**입니다. 즉, 중고자동차 매입세액공제 특례에 따라 발생한 공제를 1년 동안 이월하여 사용할 수 있습니다.



## 6. 대화 이력 확인

In [10]:
# 현재 세션의 대화 이력 출력
history = get_session_history(session_id)
print("📜 대화 이력:")
print("=" * 60)
for i, message in enumerate(history.messages, 1):
    role = "👤 User" if message.type == "human" else "🤖 AI"
    print(f"[{i}] {role}: {message.content[:100]}...\n")

📜 대화 이력:
[1] 👤 User: 2026년에 바뀌는 세액공제 제도를 종류별로 정리해줘...

[2] 🤖 AI: 2026년에 바뀌는 세액공제 제도는 다음과 같이 정리할 수 있습니다:

1. **중고자동차 매입세액공제 특례**
   - **주요 내용**: 중고자동차에 대한 공제한도를 신설합니다...

[3] 👤 User: 그 중 시행일이 7월 1일 이후인건 뭐야?...

[4] 🤖 AI: 2026년 7월 1일 이후에 시행되는 세액공제 제도는 **중고자동차 매입세액공제 특례**입니다. 이 제도는 중고자동차에 대한 공제한도를 신설하고, 과세표준에서 세금계산서를 발급받고...

[5] 👤 User: 이월 공제가 허용되는 기간은?...

[6] 🤖 AI: 이월 공제가 허용되는 기간은 **2개 과세기간(1년)**입니다. 즉, 중고자동차 매입세액공제 특례에 따라 발생한 공제를 1년 동안 이월하여 사용할 수 있습니다....



## 7. 다중 세션 테스트

서로 다른 세션은 독립적인 대화 이력을 유지합니다.

In [11]:
# 두 번째 사용자 (다른 세션) — 완전히 다른 정책을 질문
session_id_2 = "user_002"
config_2 = {"configurable": {"session_id": session_id_2}}

print("=" * 60)
print("새로운 사용자 (user_002) 세션 시작")
print("=" * 60 + "\n")

question_new = "장애인 표준사업장 세액감면이 2026년에 어떻게 달라지나요?"
print(f"👤 User (002): {question_new}")
answer_new = conversational_rag_chain.invoke({"question": question_new}, config=config_2)
print(f"🤖 AI: {answer_new}\n")

# user_001의 이력은 유지됨 — 세액공제 → 개인 → 자녀 가구 맥락이 살아있어야 함
print("\n" + "=" * 60)
print("user_001 세션으로 복귀")
print("=" * 60 + "\n")

question_continue = "아까 7월 1일부터 시행되는 공제 중에서 자녀가 있는 가구에 유리한 건 뭐야?"
print(f"👤 User (001): {question_continue}")
answer_continue = conversational_rag_chain.invoke({"question": question_continue}, config=config)
print(f"🤖 AI: {answer_continue}")

새로운 사용자 (user_002) 세션 시작

👤 User (002): 장애인 표준사업장 세액감면이 2026년에 어떻게 달라지나요?
🤖 AI: 2026년부터 장애인 표준사업장에 대한 세액감면이 확대됩니다. 구체적으로, 소득세와 법인세 감면율이 다음과 같이 변경됩니다:

- **종전**: 3년 100% + 2년 50%
- **개정**: 3년 100% + 2년 50% + 5년 30%

이 개정 내용은 2026년 1월 1일 이후 장애인 표준사업장으로 인증받은 내국인에게 적용됩니다. 이는 장애인 일자리 창출을 지원하기 위한 조치입니다.


user_001 세션으로 복귀

👤 User (001): 아까 7월 1일부터 시행되는 공제 중에서 자녀가 있는 가구에 유리한 건 뭐야?
🤖 AI: 2026년 7월 1일부터 시행되는 공제 중에서 자녀가 있는 가구에 유리한 것은 **신용카드 등 소득공제 한도 확대**입니다. 이 제도는 기존에 자녀 수와 무관했던 신용카드 등 소득공제 기본한도를 자녀당 50만원씩 상향(최대 100만원)하고, 적용 기한을 3년 연장하는 내용입니다. 

또한, 총급여가 7천만원을 초과하는 경우에도 자녀당 25만원(최대 50만원)으로 상향됩니다. 이러한 변화는 자녀 양육 부담을 완화하는 데 기여할 것입니다.


## 8. 대화 이력 초기화

In [12]:
# 특정 세션 초기화
def clear_session(session_id: str):
    """세션의 대화 이력을 초기화합니다."""
    if session_id in store:
        store[session_id].clear()
        print(f"✅ 세션 '{session_id}' 초기화 완료")
    else:
        print(f"⚠️  세션 '{session_id}'이(가) 존재하지 않습니다.")

# 모든 세션 초기화
def clear_all_sessions():
    """모든 세션의 대화 이력을 초기화합니다."""
    store.clear()
    print("✅ 모든 세션 초기화 완료")

# 테스트
clear_session("user_001")
print(f"\n현재 활성 세션: {list(store.keys())}")

✅ 세션 'user_001' 초기화 완료

현재 활성 세션: ['user_001', 'user_002']


## 💡 핵심 정리

### 대화형 RAG의 구성 요소

1. **ChatMessageHistory**: 대화 이력 저장
2. **MessagesPlaceholder**: 프롬프트에 이력 삽입
3. **RunnableWithMessageHistory**: 체인에 이력 관리 추가
4. **세션 관리**: 사용자별 독립적인 대화 유지

### 실전 활용 시나리오

- **고객 지원 챗봇**: 고객과의 대화 문맥 유지
- **교육 AI 튜터**: 학습 진행 상황 기억
- **문서 탐색 도우미**: 연속된 질문으로 심화 탐구

### 대화 이력 저장 옵션

**메모리 기반 (현재)**:
```python
store = {}  # 재시작 시 초기화
```

**영구 저장 (Production)**:
```python
# SQLite, Redis, PostgreSQL 등 사용
from langchain.memory import SQLChatMessageHistory

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_history.db"
    )
```

### 주의사항

- ⚠️ **컨텍스트 길이**: 대화가 길어지면 토큰 제한 초과 가능
  - 해결: 최근 N개 메시지만 유지하거나 요약 사용
- ⚠️ **개인정보**: 대화 이력에 민감 정보 포함 주의
  - 해결: 암호화 저장, 주기적 삭제

### 성능 향상 팁

1. **대화 길이 제한**: 최근 5~10개 메시지만 유지
2. **요약 활용**: 긴 대화는 요약하여 저장
3. **세션 타임아웃**: 일정 시간 후 자동 초기화

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 구성 요소 | `RunnableWithMessageHistory` + `ChatMessageHistory` |
| 세션 관리 | `session_id`로 여러 사용자/대화를 독립적으로 관리 |
| 이력 저장 | 인메모리 딕셔너리 — 프로덕션에선 Redis/DB로 교체 권장 |
| `MessagesPlaceholder` | 프롬프트 내 대화 이력 삽입 위치 지정 |
| 주의 | 이력이 길어질수록 토큰 비용 증가 — 최근 N턴만 유지하는 전략 고려 |

### 대화형 RAG 핵심 흐름

```
1. 사용자 질문 입력
2. ChatMessageHistory에서 해당 session_id의 이력 로드
3. 이력 + 현재 질문으로 VectorStore 검색
4. 검색 결과 + 이력 + 질문 → LLM 답변 생성
5. 새 질문/답변 쌍을 ChatMessageHistory에 저장
```

### 세션 이력 관리 옵션

| 저장소 | 용도 | 장점 |
|--------|------|------|
| `ChatMessageHistory` (기본) | 단일 서버, 개발용 | 간단, 설치 불필요 |
| `RedisChatMessageHistory` | 멀티 서버, 프로덕션 | 영속성, 분산 지원 |
| `SQLChatMessageHistory` | DB 기반 저장 | 이력 조회·분석 가능 |

### 다음 노트북 예고

**04-RAPTOR** — 긴 문서를 계층적으로 요약해 레벨별 검색이 가능한 트리 구조를 만드는 RAPTOR(Recursive Abstractive Processing for Tree-Organized Retrieval) 기법을 배워요.